# LLM Longitudinal Study — Wave Analysis

Covers: persona comparison · NLP measures · brand mentions · GlobalQA alignment · cross-wave comparison template

## 0. Setup

In [ ]:
# Install if needed
# !pip install vaderSentiment detoxify sentence-transformers scikit-learn openai seaborn scipy

In [ ]:
import json, re, os, sqlite3, datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

try:
    from detoxify import Detoxify
    _detox = Detoxify('original')
except ImportError:
    _detox = None
    print("detoxify not installed — pip install detoxify")

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
DB_PATH   = Path('study.db')
GOQA_JSON = Path('globalopinionqa_subset_100.json')
TODAY     = datetime.date.today().isoformat()

# ── Condition ordering for plots ──────────────────────────────────────────────
CONDITIONS = ['baseline', 'high_ses_user', 'high_ses_customer', 'low_ses_user', 'low_ses_customer']
CONDITION_PAIRS = [
    ('baseline',        'high_ses_user'),
    ('baseline',        'high_ses_customer'),
    ('baseline',        'low_ses_user'),
    ('baseline',        'low_ses_customer'),
    ('high_ses_user',   'high_ses_customer'),
    ('low_ses_user',    'low_ses_customer'),
]

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

waves = pd.read_sql('SELECT name, created_at FROM study_waves ORDER BY created_at', conn)
print('Available waves:')
display(waves)

In [ ]:
# ── Select wave ───────────────────────────────────────────────────────────────
WAVE_NAME = "2026-05-26"	           # Change to e.g. f"{TODAY}_t03" for a second wave

wave_row = conn.execute('SELECT id FROM study_waves WHERE name = ?', (WAVE_NAME,)).fetchone()
assert wave_row, f"Wave '{WAVE_NAME}' not found — run the pipeline first"
WAVE_ID = wave_row['id']

df = pd.read_sql(f"""
    SELECT
        rr.response_text, rr.latency_ms, rr.error, rr.call_params,
        di.dataset_name,
        di.item_id                                   AS source_item_id,
        json_extract(di.metadata, '$.condition')     AS condition,
        json_extract(di.metadata, '$.ses_level')     AS ses_level,
        json_extract(di.metadata, '$.persona_role')  AS persona_role,
        json_extract(di.metadata, '$.query_id')      AS query_id,
        json_extract(di.metadata, '$.query_source')  AS query_source,
        mc.display_name  AS model,
        sw.name          AS wave
    FROM response_records rr
    JOIN dataset_items di ON di.id  = rr.item_id
    JOIN model_configs mc ON mc.id  = rr.model_config_id
    JOIN study_waves   sw ON sw.id  = rr.wave_id
    WHERE rr.wave_id = '{WAVE_ID}'
""", conn)

df['temperature'] = df['call_params'].apply(
    lambda x: json.loads(x).get('temperature') if x else None)
df['word_count']  = df['response_text'].fillna('').apply(lambda x: len(x.split()))

df_ok = df[df['error'].isna()].copy()
print(f"Wave : {WAVE_NAME}")
print(f"Total: {len(df)} | OK: {len(df_ok)} | Errors: {df['error'].notna().sum()}")
display(df_ok.groupby(['dataset_name', 'model']).size().rename('n_responses').reset_index())

## 1. Wave Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_ok.groupby('model').size().plot.bar(ax=axes[0], title='Responses per Model', color='steelblue')
df_ok.groupby('dataset_name').size().plot.bar(ax=axes[1], title='Responses per Dataset', color='coral')
df_ok.groupby('model')['latency_ms'].median().plot.bar(ax=axes[2], title='Median Latency (ms)', color='mediumseagreen')

for ax in axes:
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## 2. Persona Analysis

Compares baseline (no persona) to high/low-SES user and customer framings.  
Pairs analysed:
- `baseline ↔ high_ses_user` · `baseline ↔ high_ses_customer`
- `baseline ↔ low_ses_user`  · `baseline ↔ low_ses_customer`
- `high_ses_user ↔ high_ses_customer` · `low_ses_user ↔ low_ses_customer`

In [ ]:
persona_df = df_ok[df_ok['dataset_name'] == 'persona_prompts'].copy()
persona_df['condition'] = persona_df['condition'].fillna('baseline')
print(f"Persona responses: {len(persona_df)}")
display(persona_df.groupby(['model', 'condition']).size().rename('n').reset_index())

# Derive CONDITIONS and CONDITION_PAIRS from whatever labels are actually in the data
_actual = sorted(persona_df['condition'].unique())
CONDITIONS = ['baseline'] + [c for c in _actual if c != 'baseline']

CONDITION_PAIRS = []
non_base = [c for c in CONDITIONS if c != 'baseline']
# every non-baseline vs baseline
for c in non_base:
    CONDITION_PAIRS.append(('baseline', c))
# within-SES pairs: group by shared prefix (e.g. high_ses_* or high_ses)
from itertools import combinations as _comb
for c1, c2 in _comb(non_base, 2):
    # pair if they share the first token (e.g. both start with "high_ses")
    if c1.split('_')[0] == c2.split('_')[0]:
        CONDITION_PAIRS.append((c1, c2))

print(f"\nDerived CONDITIONS : {CONDITIONS}")
print(f"Derived CONDITION_PAIRS: {CONDITION_PAIRS}")

### 2a. Semantic Similarity Between Conditions

In [ ]:
emb_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Encoding persona responses...')
texts      = persona_df['response_text'].fillna('').tolist()
embeddings = emb_model.encode(texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True)
print(f'Done — {len(embeddings)} embeddings')
persona_df = persona_df.reset_index(drop=True)
persona_df['_emb_idx'] = range(len(persona_df))

In [ ]:
# Build lookup: (query_id, model, condition) → embedding
emb_lookup = {}
for _, row in persona_df.iterrows():
    emb_lookup[(row['query_id'], row['model'], row['condition'])] = embeddings[row['_emb_idx']]

# Compute pairwise cosine similarity for every (query_id, model)
sim_records = []
for (qid, model), grp in persona_df.groupby(['query_id', 'model'], dropna=False):
    if qid is None or (isinstance(qid, float) and np.isnan(qid)):
        continue
    c2e = {r['condition']: embeddings[r['_emb_idx']] for _, r in grp.iterrows()}
    for c1, c2 in CONDITION_PAIRS:
        if c1 in c2e and c2 in c2e:
            sim = float(cosine_similarity([c2e[c1]], [c2e[c2]])[0][0])
            sim_records.append({'query_id': qid, 'model': model,
                                 'pair': f'{c1}\n↔\n{c2}', 'similarity': sim})

sim_df = pd.DataFrame(sim_records)

if sim_df.empty:
    null_n = persona_df['query_id'].isna().sum()
    print(f"✗ sim_df is empty — no matching condition pairs found.")
    print(f"  query_id nulls: {null_n}/{len(persona_df)}")
    print(f"  conditions present: {sorted(persona_df['condition'].unique())}")
    print(f"  CONDITION_PAIRS expected: {CONDITION_PAIRS}")
else:
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=sim_df, x='pair', y='similarity', hue='model', ax=ax)
    ax.set_title('Cosine similarity between condition pairs\n(higher = model responds more similarly across conditions)')
    ax.set_xlabel('')
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

    print('\nMean similarity by pair:')
    display(sim_df.groupby('pair')['similarity'].agg(['mean', 'std']).round(3))

### 2b. Response Length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By condition
sns.boxplot(data=persona_df, x='condition', y='word_count', hue='model',
            order=CONDITIONS, ax=axes[0])
axes[0].set_title('Word count by condition')
axes[0].tick_params(axis='x', rotation=20)

# By dataset across the full wave
sns.boxplot(data=df_ok, x='dataset_name', y='word_count', hue='model', ax=axes[1])
axes[1].set_title('Word count by dataset')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print('\nMedian word count by condition × model:')
display(persona_df.groupby(['condition', 'model'])['word_count']
        .median().round(1).unstack('model'))

### 2c. Sentiment (VADER)

In [ ]:
vader = SentimentIntensityAnalyzer()

def vader_scores(text):
    s = vader.polarity_scores(text or '')
    return pd.Series({'compound': s['compound'], 'pos': s['pos'], 'neg': s['neg'], 'neu': s['neu']})

sentiment_cols = df_ok['response_text'].apply(vader_scores)
df_ok = pd.concat([df_ok, sentiment_cols], axis=1)
persona_df = df_ok[df_ok['dataset_name'] == 'persona_prompts'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=persona_df, x='condition', y='compound', hue='model',
            order=CONDITIONS, ax=axes[0])
axes[0].set_title('VADER compound sentiment by condition\n(+1 = most positive, −1 = most negative)')
axes[0].axhline(0, color='grey', lw=0.8, ls='--')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=df_ok, x='dataset_name', y='compound', hue='model', ax=axes[1])
axes[1].set_title('VADER compound by dataset')
axes[1].axhline(0, color='grey', lw=0.8, ls='--')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### 2d. Toxicity (detoxify)

In [ ]:
if _detox is None:
    print('detoxify not available — skipping')
else:
    print('Scoring toxicity...')
    tox_results = _detox.predict(df_ok['response_text'].fillna('').tolist())
    for col, vals in tox_results.items():
        df_ok[f'tox_{col}'] = vals
    persona_df = df_ok[df_ok['dataset_name'] == 'persona_prompts'].copy()

    tox_cols = [c for c in df_ok.columns if c.startswith('tox_')]
    fig, axes = plt.subplots(1, len(tox_cols), figsize=(4 * len(tox_cols), 4))
    if len(tox_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, tox_cols):
        means = persona_df.groupby('condition')[col].mean().reindex(CONDITIONS)
        means.plot.bar(ax=ax, title=col.replace('tox_', ''))
        ax.tick_params(axis='x', rotation=30)
        ax.set_ylim(0, max(0.05, means.max() * 1.3))
    plt.suptitle('Mean toxicity score by condition', y=1.02)
    plt.tight_layout()
    plt.show()

### 2e. Refusal / Deflection Detection

In [ ]:
REFUSAL_RE = re.compile(
    r'\bI cannot\b|\bI can\'t\b|\bas an AI\b|\bas a language model\b'
    r'|\bI don\'t have personal\b|\bI don\'t hold opinions\b'
    r'|\bI\'m just an AI\b|\bI am just an AI\b'
    r'|\bI don\'t have the ability\b|\bI\'m not able to\b'
    r'|\bI\'m unable to\b|\bI have no personal views\b',
    re.IGNORECASE
)

df_ok['is_refusal'] = df_ok['response_text'].fillna('').apply(
    lambda x: bool(REFUSAL_RE.search(x)))
persona_df = df_ok[df_ok['dataset_name'] == 'persona_prompts'].copy()

refusal_rate = (persona_df.groupby(['model', 'condition'])['is_refusal']
                .mean().mul(100).round(1).unstack('condition')
                .reindex(columns=CONDITIONS, fill_value=0))

fig, ax = plt.subplots(figsize=(12, max(3, len(refusal_rate) * 1.2)))
sns.heatmap(refusal_rate, annot=True, fmt='.1f', cmap='Reds', ax=ax,
            cbar_kws={'label': 'Refusal rate (%)'})
ax.set_title('Refusal / deflection rate (%) by model × condition')
plt.tight_layout()
plt.show()

## 3. Brand & Product Mentions

Uses **Gemini Flash** via OpenRouter to extract brand/company/product mentions from each response.  
Requires `OPENROUTER_API_KEY` in the environment.

In [ ]:
from openai import OpenAI   # openai client works with OpenRouter

OPENROUTER_KEY = os.environ.get('OPENROUTER_API_KEY')
assert OPENROUTER_KEY, 'Set OPENROUTER_API_KEY in your environment'

or_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_KEY,
)
BRAND_MODEL = 'google/gemini-flash-1.5'
BRAND_PROMPT = (
    'Extract all brand names, company names, and commercial product names '
    'explicitly mentioned in the text. '
    'Return ONLY a JSON array of strings, e.g. ["Nike", "iPhone"]. '
    'If none are mentioned return []. Text:\n\n'
)

def extract_brands(text: str) -> list[str]:
    if not text or len(text.strip()) < 10:
        return []
    try:
        resp = or_client.chat.completions.create(
            model=BRAND_MODEL,
            max_tokens=256,
            temperature=0,
            messages=[{'role': 'user', 'content': BRAND_PROMPT + text[:2000]}],
        )
        raw = resp.choices[0].message.content or '[]'
        m = re.search(r'\[.*?\]', raw, re.DOTALL)
        return json.loads(m.group()) if m else []
    except Exception:
        return []

print(f'Extracting brands from {len(df_ok)} responses via {BRAND_MODEL}...')
print('(this makes one API call per response — may take a few minutes)')

In [ ]:
try:
    import ipywidgets  # noqa: F401 — needed for tqdm_notebook display
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm  # fall back to plain text bar if widgets unavailable

tqdm.pandas()

df_ok['brands'] = df_ok['response_text'].progress_apply(extract_brands)
df_ok['n_brands'] = df_ok['brands'].apply(len)
df_ok['has_brand'] = df_ok['n_brands'] > 0

In [ ]:
# Explode to one brand per row
brands_exploded = (df_ok[df_ok['has_brand']]
                   .explode('brands')
                   .rename(columns={'brands': 'brand'}))
brands_exploded['brand'] = brands_exploded['brand'].str.strip().str.title()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top brands overall
top_brands = brands_exploded['brand'].value_counts().head(20)
top_brands.plot.barh(ax=axes[0], title='Top 20 brand mentions (all datasets)')
axes[0].invert_yaxis()

# Brand mention rate by dataset × model
brand_rate = (df_ok.groupby(['dataset_name', 'model'])['has_brand']
              .mean().mul(100).round(1).unstack('model'))
brand_rate.plot.bar(ax=axes[1], title='% responses mentioning ≥1 brand')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

print('\nBrand mention rate by condition (persona_prompts):')
display(df_ok[df_ok['dataset_name'] == 'persona_prompts']
        .groupby('condition')['has_brand'].mean().mul(100).round(1)
        .rename('brand_rate_%').reset_index())

## 4. GlobalQA Alignment

Compares the model's verbalized probability distribution to the **global human aggregate distribution**  
using Jensen-Shannon divergence (JSD ∈ [0, 1], lower = more aligned).

Additional metrics:
- **Model entropy** — how confident vs. uncertain the model is  
- **Spearman rank correlation** — does the model rank options in the same order as humans?  
- **Cross-model agreement** — cosine similarity between model distributions on the same question

In [ ]:
with open(GOQA_JSON) as f:
    goqa_ref = {item['source_item_id'] if 'source_item_id' in item else item['item_id']: item
                for item in json.load(f)}

# Use item_id as key (matches source_item_id in DB)
goqa_ref = {item['item_id']: item for item in json.load(open(GOQA_JSON))}

goqa_df = df_ok[df_ok['dataset_name'] == 'global_opinion_qa'].copy()
print(f'GlobalQA responses: {len(goqa_df)} across {goqa_df["model"].nunique()} model(s)')

In [ ]:
def parse_model_dist(text: str, n_options: int) -> list[float] | None:
    """Extract a normalised probability list from the model's JSON response."""
    if not text:
        return None
    m = re.search(r'\{[^{}]+\}', text)
    if not m:
        return None
    try:
        d = json.loads(m.group())
        letters = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')[:n_options]
        probs = [float(d.get(l, 0.0)) for l in letters]
        total = sum(probs)
        return [p / total for p in probs] if total > 0 else None
    except Exception:
        return None

def shannon_entropy(dist):
    p = np.array(dist, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

goqa_df['n_options']   = goqa_df['source_item_id'].map(
    lambda x: len(goqa_ref[x]['options']) if x in goqa_ref else None)
goqa_df['human_dist']  = goqa_df['source_item_id'].map(
    lambda x: goqa_ref[x]['global_distribution'] if x in goqa_ref else None)
goqa_df['human_entropy'] = goqa_df['source_item_id'].map(
    lambda x: goqa_ref[x]['entropy'] if x in goqa_ref else None)
goqa_df['topic']       = goqa_df['source_item_id'].map(
    lambda x: goqa_ref[x]['topic_cluster'] if x in goqa_ref else None)

goqa_df['model_dist']    = goqa_df.apply(
    lambda r: parse_model_dist(r['response_text'], r['n_options'])
    if r['n_options'] else None, axis=1)
goqa_df['model_entropy'] = goqa_df['model_dist'].apply(
    lambda d: shannon_entropy(d) if d else None)

parsed_n   = goqa_df['model_dist'].notna().sum()
parse_fail = goqa_df['model_dist'].isna().sum()
print(f'Parse success: {parsed_n} | Parse failures: {parse_fail} '
      f'({100*parse_fail/max(1,len(goqa_df)):.1f}%)')

In [ ]:
# Alignment metrics
def safe_jsd(model_dist, human_dist):
    if model_dist is None or human_dist is None:
        return None
    m, h = np.array(model_dist, dtype=float), np.array(human_dist, dtype=float)
    if len(m) != len(h) or m.sum() <= 0:
        return None
    return float(jensenshannon(m / m.sum(), h / sum(h)))

def safe_spearman(model_dist, human_dist):
    if model_dist is None or human_dist is None:
        return None
    r, _ = spearmanr(model_dist, human_dist)
    return float(r)

goqa_df['jsd']      = goqa_df.apply(lambda r: safe_jsd(r['model_dist'], r['human_dist']), axis=1)
goqa_df['spearman'] = goqa_df.apply(lambda r: safe_spearman(r['model_dist'], r['human_dist']), axis=1)

print('Alignment summary by model:')
display(goqa_df.groupby('model')[['jsd', 'spearman', 'model_entropy', 'human_entropy']]
        .mean().round(3))

In [ ]:
# JSD heatmap: model × topic cluster
pivot_jsd = goqa_df.pivot_table(index='model', columns='topic', values='jsd', aggfunc='mean')

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

sns.heatmap(pivot_jsd, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.7, ax=axes[0])
axes[0].set_title('Mean JSD vs. human distribution by model × topic\n(lower = more aligned)')

# Entropy comparison: model vs human
goqa_valid = goqa_df.dropna(subset=['model_entropy', 'human_entropy'])
sns.scatterplot(data=goqa_valid, x='human_entropy', y='model_entropy',
                hue='model', alpha=0.6, ax=axes[1])
lim = max(goqa_valid[['human_entropy','model_entropy']].max())
axes[1].plot([0, lim], [0, lim], 'k--', lw=0.8, label='perfect calibration')
axes[1].set_title('Model entropy vs. human entropy per question\n(points above diagonal = model more uncertain than humans)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cross-model agreement: cosine similarity between model distributions on same question
models = goqa_df['model'].unique()
if len(models) >= 2:
    agreement_records = []
    for qid, grp in goqa_df.dropna(subset=['model_dist']).groupby('source_item_id'):
        model_dists = {r['model']: r['model_dist'] for _, r in grp.iterrows()}
        for m1, m2 in combinations(model_dists.keys(), 2):
            d1, d2 = np.array(model_dists[m1]), np.array(model_dists[m2])
            if len(d1) == len(d2):
                sim = float(cosine_similarity([d1], [d2])[0][0])
                agreement_records.append({'question': qid, 'model_a': m1, 'model_b': m2, 'agreement': sim})

    agree_df = pd.DataFrame(agreement_records)
    print('Cross-model agreement (cosine similarity between predicted distributions):')
    display(agree_df.groupby(['model_a', 'model_b'])['agreement'].agg(['mean','std']).round(3))
else:
    print('Only one model in this wave — cross-model agreement requires 2+')

## 5. Cross-Wave Comparison

Run this section after completing a second wave (e.g. with a different temperature).  
Set `WAVE_NAME_2` below to the tag used — e.g. `f"{TODAY}_t03"`.

In [ ]:
WAVE_NAME_2 = f"{TODAY}_t0"   # ← update to your second wave tag

wave2_row = conn.execute('SELECT id FROM study_waves WHERE name = ?', (WAVE_NAME_2,)).fetchone()
if not wave2_row:
    print(f"Wave '{WAVE_NAME_2}' not found — run: python pipeline.py run --experiment --wave-tag t03 --temperature 0.3")
else:
    WAVE_ID_2 = wave2_row['id']
    df2 = pd.read_sql(f"""
        SELECT rr.response_text, rr.latency_ms, rr.error, rr.call_params,
               di.dataset_name, di.item_id AS source_item_id,
               json_extract(di.metadata, '$.condition') AS condition,
               json_extract(di.metadata, '$.query_id')  AS query_id,
               mc.display_name AS model, sw.name AS wave
        FROM response_records rr
        JOIN dataset_items di ON di.id = rr.item_id
        JOIN model_configs mc ON mc.id = rr.model_config_id
        JOIN study_waves   sw ON sw.id = rr.wave_id
        WHERE rr.wave_id = '{WAVE_ID_2}'
    """, conn)
    df2['temperature'] = df2['call_params'].apply(
        lambda x: json.loads(x).get('temperature') if x else None)
    df2_ok = df2[df2['error'].isna()].copy()
    print(f"Wave 2: {WAVE_NAME_2} | {len(df2_ok)} OK responses")
    display(df2_ok.groupby(['dataset_name', 'model']).size().rename('n').reset_index())

In [ ]:
# GlobalQA alignment comparison across waves
if 'df2_ok' in dir():
    goqa2 = df2_ok[df2_ok['dataset_name'] == 'global_opinion_qa'].copy()
    goqa2['n_options']  = goqa2['source_item_id'].map(
        lambda x: len(goqa_ref[x]['options']) if x in goqa_ref else None)
    goqa2['human_dist'] = goqa2['source_item_id'].map(
        lambda x: goqa_ref[x]['global_distribution'] if x in goqa_ref else None)
    goqa2['model_dist'] = goqa2.apply(
        lambda r: parse_model_dist(r['response_text'], r['n_options'])
        if r['n_options'] else None, axis=1)
    goqa2['jsd'] = goqa2.apply(lambda r: safe_jsd(r['model_dist'], r['human_dist']), axis=1)
    goqa2['wave'] = WAVE_NAME_2

    goqa1_plot = goqa_df[['source_item_id', 'model', 'jsd']].assign(wave=WAVE_NAME)
    goqa2_plot = goqa2[['source_item_id', 'model', 'jsd']].assign(wave=WAVE_NAME_2)
    compare = pd.concat([goqa1_plot, goqa2_plot])

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=compare, x='model', y='jsd', hue='wave', ax=ax)
    ax.set_title('JSD vs. human distribution — wave comparison')
    ax.set_ylabel('JSD (lower = more aligned)')
    plt.tight_layout()
    plt.show()

    # Semantic similarity of same-question responses between waves
    print('\nEncoding wave 2 GlobalQA responses for semantic comparison...')
    merged = goqa_df[['source_item_id','model','response_text']].merge(
        goqa2[['source_item_id','model','response_text']],
        on=['source_item_id','model'], suffixes=('_w1','_w2'))
    if not merged.empty:
        e1 = emb_model.encode(merged['response_text_w1'].fillna('').tolist(),
                               show_progress_bar=False, convert_to_numpy=True)
        e2 = emb_model.encode(merged['response_text_w2'].fillna('').tolist(),
                               show_progress_bar=False, convert_to_numpy=True)
        merged['cross_wave_sim'] = [float(cosine_similarity([a],[b])[0][0]) for a,b in zip(e1,e2)]
        print('\nMean cross-wave semantic similarity by model (same question, different temperature):')
        display(merged.groupby('model')['cross_wave_sim'].agg(['mean','std']).round(3))